# `JAXsim` Showcase: Parallel Simulation of a free-falling body

First, we install the necessary packages and import them.

In [ ]:
# @title Imports and setup
import sys
import os
import pathlib

# Deactivate GPU to avoid out of memory errors
os.environ["CUDA_VISIBLE_DEVICES"] = ""

from IPython.display import HTML, clear_output, display

# Install JAX and Gazebo

# Set environment variable to avoid GPU out of memory errors
%env XLA_PYTHON_CLIENT_MEM_PREALLOCATE=false

import time
from typing import Dict, Tuple

import jax
import jax.numpy as jnp
import jax_dataclasses
import rod
from rod.builder.primitives import SphereBuilder, BoxBuilder

import jaxsim.typing as jtp
from jaxsim import logging
from jaxsim.api.common import VelRepr
from jaxsim.rbda.collidable_points import collidable_points_pos_vel
from jaxsim.api.model import JaxSimModel
from jaxsim.api.data import JaxSimModelData

from jaxsim.mujoco import (
    MujocoVideoRecorder,
    MujocoModelHelper,
    RodModelToMjcf,
    SdfToMjcf,
    UrdfToMjcf,
)

logging.set_logging_level(logging.LoggingLevel.INFO)
logging.info(f"Running on {jax.devices()}")

We will use a simple sphere model to simulate a free-falling body. The spheres set will be composed of 9 spheres, each with a different position. The spheres will be simulated in parallel, and the simulation will be run for 3000 steps corresponding to 3 seconds of simulation.

**Note**: Parallel simulations are independent of each other, the different position is imposed only to show the parallelization visually.

In [ ]:
# rod_model = rod.Sdf(
#     version="1.7",
#     model=rod.Model(
#         name="single_pendulum",
#         link=[
#             rod.Link(
#                 name="base_link",
#                 inertial=rod.Inertial(mass=100.0, inertia=rod.Inertia()),
#                 collision=rod.Collision(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 2.15])),
#                     pose=rod.Pose(pose=[0, 0, 1, 0, 0, 0]),
#                     name="base_link_fixed_joint_lump__base_collision",
#                 ),
#                 visual=rod.Visual(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 2.15])),
#                     pose=rod.Pose(pose=[0, 0, 1, 0, 0, 0]),
#                     name="base_link_fixed_joint_lump__base_visual",
#                 ),
#             ),
#             rod.Link(
#                 name="upper_link",
#                 pose=rod.Pose(pose=[0, 0, 0, 0, 0, 0], relative_to="upper_joint"),
#                 inertial=rod.Inertial(
#                     mass=0.5,
#                     inertia=rod.Inertia(),
#                     pose=rod.Pose([0, 0, 0.5, 0, 0, 0]),
#                 ),
#                 collision=rod.Collision(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 1.0])),
#                     pose=rod.Pose(pose=[0, 0, 0.5, 0, 0, 0]),
#                     name="upper_link_collision",
#                 ),
#                 visual=rod.Visual(
#                     geometry=rod.Geometry(rod.Box(size=[0.15, 0.15, 1.0])),
#                     pose=rod.Pose(pose=[0, 0, 0.5, 0, 0, 0]),
#                     name="upper_link_visual",
#                 ),
#             ),
#         ],
#         joint=[
#             rod.Joint(
#                 name="upper_joint",
#                 type="revolute",
#                 parent="base_link",
#                 child="upper_link",
#                 axis=rod.Axis(
#                     xyz=rod.Xyz([1, 0, 0]),
#                     limit=rod.Limit(lower=-5, upper=5),
#                 ),
#                 pose=rod.Pose(
#                     pose=[0.15, 0, 2.0, -1.5708, 0, 0], relative_to="base_link"
#                 ),
#             ),
#         ],
#     ),
# )

# model_sdf_string = rod.urdf.exporter.UrdfExporter.sdf_to_urdf_string(
#     sdf=rod_model.models()[0]
# )

In [ ]:
# @title Create a sphere model
model_sdf_string = rod.Sdf(
    version="1.7",
    model=BoxBuilder(x=0.30, y=0.30, z=1.0, mass=1.0, name="box")
    # model=SphereBuilder(radius=0.15, mass=1.0, name="sphere")
    .build_model().add_link().add_inertial().add_visual().add_collision().build(),
).serialize(pretty=True)

# import urllib

# url = "https://raw.githubusercontent.com/icub-tech-iit/ergocub-gazebo-simulations/master/models/stickBot/model.urdf"

# model_sdf_string = urllib.request.urlopen(url).read().decode()
# model_sdf_string = pathlib.Path("/home/flferretti/git/element_rl-for-codesign/assets/model/hopper.sdf")

JAXsim offers a simple functional API in order to interact in a memory-efficient way with the simulation. Four main elements are used to define a simulation:

- `model`: an object that defines the dynamics of the system.
- `data`: an object that contains the state of the system.
- `integrator`: an object that defines the integration method.
- `integrator_state`: an object that contains the state of the integrator.

In [ ]:
import jaxsim.api as js
from jaxsim import integrators
import jaxsim
import jaxopt

dt = 0.001
integration_time = 3000

model = js.model.JaxSimModel.build_from_model_description(
    model_description=model_sdf_string,
    contact_model=js.rigid_contacts.RigidContacts(),
    is_urdf=True,
)

# model = js.model.reduce(
#     model=model,
#     considered_joints=tuple(
#         [
#             j
#             for j in model.joint_names()
#             if "camera" not in j
#             # and "neck" not in j
#             # and "wrist" not in j
#             # and "thumb" not in j
#             # and "index" not in j
#             # and "middle" not in j
#             # and "ring" not in j
#             # and "pinkie" not in j
#             # and "elbow" not in j
#             # and "shoulder" not in j
#             # and "hip" not in j
#             # and "knee" not in j
#             and "lidar" not in j and "torso" not in j
#         ]
#     ),
# )
# model = js.model.reduce(
#     model=model,
#     considered_joints=(
#         "l_hip_pitch",  # 0
#         "l_shoulder_pitch",  # 1
#         "r_hip_pitch",  # 2
#         "r_shoulder_pitch",  # 3
#         "l_hip_roll",  # 4
#         "l_shoulder_roll",  # 5
#         "r_hip_roll",  # 6
#         "r_shoulder_roll",  # 7
#         "l_hip_yaw",  # 8
#         "l_shoulder_yaw",  # 9
#         "r_hip_yaw",  # 10
#         "r_shoulder_yaw",  # 11
#         "l_knee",  # 12
#         "l_elbow",  # 13
#         "r_knee",  # 14
#         "r_elbow",  # 15
#         "l_ankle_pitch",  # 16
#         "r_ankle_pitch",  # 17
#         "l_ankle_roll",  # 18
#         "r_ankle_roll",  # 19
#     ),
# )

# model = js.model.reduce(model=model, considered_joints=tuple())

data = js.data.JaxSimModelData.build(
    model=model, velocity_representation=VelRepr.Inertial
)
integrator = integrators.fixed_step.RungeKutta4SO3.build(
    dynamics=js.ode.wrap_system_dynamics_for_integration(
        model=model,
        data=data,
        system_dynamics=js.ode.system_dynamics,
    ),
)
# with jax.disable_jit():
integrator_state = integrator.init(x0=data.state, t0=0.0, dt=dt)

In [ ]:
# camera = {
#     "name": "pendulum_camera",
#     "mode": "fixed",
#     "pos": "3.586 0.587 3.432",
#     "xyaxes": "-0.162 0.987 -0.000 -0.514 -0.084 0.854",
#     "fovy": "60",
# }

mcjf_string, assets = UrdfToMjcf.convert(urdf=model_sdf_string)  # , cameras=camera)
mj_helper = MujocoModelHelper.build_from_xml(
    mjcf_description=mcjf_string, assets=assets
)
recorder = MujocoVideoRecorder(
    model=mj_helper.model, data=mj_helper.data, fps=int(1 / dt), width=640, height=480
)

It is possible to automatically choose a good set of parameters for the terrain. 

By default, in JaxSim a sphere primitive has 250 collision points. This can be modified by setting the `JAXSIM_COLLISION_SPHERE_POINTS` environment variable.

Given that at its steady-state the sphere will act on two or three points, we can estimate the ground parameters by explicitly setting the number of active points to these values.

In [ ]:
# data = data.replace(
#     soft_contacts_params=js.contact.estimate_good_soft_contacts_parameters(
#         model, number_of_active_collidable_points_steady_state=3
#     )
# )
import jaxlie

data = data.reset_base_quaternion(
    base_quaternion=jaxlie.SO3.from_rpy_radians(
        jnp.pi / 12, 0.0, 0.0
    ).as_quaternion_xyzw()[jnp.array([3, 0, 1, 2])]
)

Let's create a position vector for a 3x3 grid. Every sphere will be placed at a different height.

In [ ]:
# Primary Calculations
envs_per_row = 1  # @slider(2, 10, 1)
initial_height = 0.5
env_spacing = 0.5
edge_len = env_spacing * (2 * envs_per_row - 1)


# Create Grid
def grid(edge_len, envs_per_row):
    edge = jnp.linspace(-edge_len, edge_len, envs_per_row)
    xx, yy = jnp.meshgrid(edge, edge)

    poses = [
        [[xx[i, j], yy[i, j], initial_height], [0, 0, 0]]
        for i in range(xx.shape[0])
        for j in range(yy.shape[0])
    ]

    return jnp.array(poses)


logging.info(f"Simulating {envs_per_row**2} environments")
poses = grid(edge_len, envs_per_row)

In order to parallelize the simulation, we first need to define a function `simulate` for a single element of the batch.

In [ ]:
model.kin_dyn_parameters.contact_parameters.point.shape

In [ ]:
# Define a function to simulate a single model instance
def simulate(
    data: js.data.JaxSimModelData, integrator_state: dict, pose: jnp.array
) -> tuple:

    def compute_force(model, data, position, velocity):
        J, a_ref, r = _contact_jacobian(model=model, data=data)

        τ = jnp.zeros_like(data.joint_positions())
        τ_constraints = jnp.zeros_like(data.joint_positions())

        S = jnp.block([jnp.zeros(shape=(model.dofs(), 6)), jnp.eye(model.dofs())]).T
        h = js.model.free_floating_bias_forces(model=model, data=data)

        R = jnp.diag(r)

        # Compute the smooth contact force.
        qf_smooth = S @ (jnp.atleast_1d(τ - τ_constraints)) - h

        M_inv = jnp.linalg.inv(
            js.model.free_floating_mass_matrix(model=model, data=data)
        )

        # C_X_L = jax.vmap(
        #     lambda position: jaxsim.math.Adjoint.from_rotation_and_translation(
        #         rotation=jnp.eye(3),
        #         translation=position,
        #     ).T
        # )(jnp.array(model.kin_dyn_parameters.contact_parameters.point))

        # points_velocities = jnp.hstack(
        #     jax.vmap(
        #         lambda C_X_Li: C_X_Li
        #         @ js.link.velocity(
        #             model=model,
        #             data=data,
        #             link_index=base,
        #         )
        #     )(C_X_L)
        # )

        # Calculate quantities for the linear optimization problem.
        A = J @ M_inv @ J.T + R
        b = J @ M_inv @ qf_smooth + jnp.hstack(velocity) - a_ref

        objective = lambda x: jnp.sum(0.5 * (A @ x + b) ** 2)

        # Compute the 3D linear force in C[W] frame
        opt = jaxopt.ProjectedGradient(
            fun=objective,
            projection=jaxopt.projection.projection_non_negative,
            maxiter=100,
            implicit_diff=False,
            maxls=20,
            verbose=2,
        )

        W_f_Ci = opt.run(jnp.zeros_like(b)).params.reshape(-1, 3)

        W_f_Ci6D = jnp.hstack([W_f_Ci, jnp.zeros_like(W_f_Ci)])

        W_f_Li_terrain = jax.vmap(
            lambda nc: (
                jnp.vstack(
                    jnp.equal(
                        jnp.array(model.kin_dyn_parameters.contact_parameters.body),
                        nc,
                    ).astype(int)
                )
                * W_f_Ci6D
            ).sum(axis=0)
        )(jnp.arange(model.number_of_links()))

        # F = J.T @ W_f_Ci

        return W_f_Li_terrain.squeeze()

    def _imp_aref(
        position: jtp.Array, velocity: jtp.Array
    ) -> tuple[jtp.Array, jtp.Array]:
        """Calculates impedance and offset acceleration in constraint frame.

        Args:
            position: position in constraint frame
            velocity: velocity in constraint frame

        Returns:
            impedance: constraint impedance
            a_ref: offset acceleration in constraint frame
        """

        imp_x = jnp.abs(position) / width
        imp_a = (1.0 / jnp.power(mid, p - 1)) * jnp.power(imp_x, p)

        imp_b = 1 - (1.0 / jnp.power(1 - mid, p - 1)) * jnp.power(1 - imp_x, p)

        imp_y = jnp.where(imp_x < mid, imp_a, imp_b)

        imp = jnp.clip(ξ_min + imp_y * (ξ_max - ξ_min), ξ_min, ξ_max)
        imp = jnp.atleast_1d(jnp.where(imp_x > 1.0, ξ_max, imp))

        # TODO: When passing negative values, K and D represent a spring and damper, respectively.
        # K = jnp.where(
        #     stiffness <= 0, -stiffness / ξ_max**2, 1 / (ξ_max * timeconst * ζ) ** 2
        # )
        # D = jnp.where(damping <= 0, -damping / ξ_max, 2 / (ξ_max * timeconst))

        K = 1 / (ξ_max * timeconst * ζ) ** 2
        D = 2 / (ξ_max * timeconst)

        a_ref = jnp.atleast_1d(D * velocity + K * imp * position)

        return imp, a_ref

    def _contact_jacobian(model: JaxSimModel, data: JaxSimModelData) -> tuple:
        """Compute the contact jacobian and the reference acceleration.

        Args:
            model (JaxSimModel): The jaxsim model.
            position (jtp.Vector): The position of the collidable point.

        Returns:
            tuple: A tuple containing the contact jacobian, the reference acceleration, and the contact radius.
        """

        def _compute_row(
            link_idx: jtp.Float, position: jtp.Array, velocity: jtp.Array
        ) -> tuple[jtp.Array, jtp.Array, jtp.Array, jtp.Array]:

            # TODO: This should be the contact frame Jacobian.
            L_Xv_C = jaxsim.math.Adjoint.from_rotation_and_translation(
                rotation=jnp.eye(3),
                translation=model.kin_dyn_parameters.contact_parameters.point[link_idx],
            ).T

            J = (
                L_Xv_C
                @ js.model.generalized_free_floating_jacobian(
                    model=model, data=data, output_vel_repr=VelRepr.Inertial
                )[link_idx]
            )[:3]

            # J_contact = δ̇  @ (friction * jnp.array([0,0,1]))

            # Compute the reference acceleration.
            imp, a_ref = _imp_aref(position=position, velocity=velocity)

            # Compute the regularization terms.
            R = (
                (2 * friction**2 * (1 - imp) / (imp + 1e-8))
                * (1 + friction**2)
                @ jnp.linalg.inv(
                    js.link.spatial_inertia(model=model, link_index=link_idx)[:3, :3]
                )
            )

            return jax.tree.map(lambda x: x * (position[2] < 0), (J, a_ref, R))

        J, a_ref, R = jax.tree.map(
            jnp.concatenate,
            jax.vmap(_compute_row)(
                jnp.array(model.kin_dyn_parameters.contact_parameters.body),
                position,
                velocity,
            ),
        )
        return J, a_ref, R

    data = data.reset_base_position(base_position=pose)
    x_t_i = []
    joint_positions = []
    forces = []
    link_positions = []
    base_orientations = []

    # l_foot = model.link_names().index("l_ankle_2")
    # r_foot = model.link_names().index("r_ankle_2")
    base = model.link_names().index("box_link")
    # base = model.link_names().index("base_link")
    # base = model.link_names().index("root_link")

    get_collidable_points_pos_vel = lambda base_position, base_quaternion, joint_positions, base_linear_velocity, base_angular_velocity, joint_velocities: collidable_points_pos_vel(
        model=model,
        base_position=base_position,
        base_quaternion=base_quaternion,
        joint_positions=joint_positions,
        base_linear_velocity=base_linear_velocity,
        base_angular_velocity=base_angular_velocity,
        joint_velocities=joint_velocities,
    )

    compute_force_jit = lambda data, position, velocity: compute_force(
        model=model, data=data, position=position, velocity=velocity
    )

    for _ in range(integration_time):
        # Check https://mujoco.readthedocs.io/en/latest/modeling.html#solver-parameters
        timeconst, ζ, ξ_min, ξ_max, width, mid, p, friction = (
            0.1,
            0.5,
            0.0,
            1.0,
            0.1,
            0.5,
            1.0,
            0.5,
        )

        position, velocity = get_collidable_points_pos_vel(
            data.base_position(),
            data.base_orientation(dcm=False),
            data.joint_positions(),
            data.base_velocity()[0:3],
            data.base_velocity()[3:6],
            data.joint_velocities(),
        )

        F = compute_force_jit(data, position, velocity)

        link_forces = jnp.zeros((model.number_of_links(), 6)).at[base].set(F)

        data, integrator_state = js.model.step(
            dt=dt,
            model=model,
            data=data,
            integrator=integrator,
            integrator_state=integrator_state,
            joint_forces=None,
            link_forces=None,
        )

        x_t_i.append(data.base_position())
        joint_positions.append(data.joint_positions())
        base_orientations.append(data.base_orientation())
        forces.append(F)
        link_positions.append(
            js.link.transform(model=model, data=data, link_index=base)[3, :3]
        )

        print(f"Simulated time: {_ * dt:.3f}s", end="\r")

    return x_t_i, base_orientations, joint_positions, forces, link_positions

We will make use of `jax.vmap` to simulate multiple models in parallel. This is a very powerful feature of JAX that allows to write code that is very similar to the single-model case, but can be executed in parallel on multiple models.
In order to do so, we need to first apply `jax.vmap` to the `simulate` function, and then call the resulting function with the batch of different poses as input.

Note that in our case we are vectorizing over the `pose` argument of the function `simulate`, this correspond to the value assigned to the `in_axes` parameter of `jax.vmap`:

`in_axes=(None, None, 0)` means that the first two arguments of `simulate` are not vectorized, while the third argument is vectorized over the zero-th dimension.

In [ ]:
# Define a function to simulate multiple model instances
# simulate_vectorized = jax.vmap(simulate, in_axes=(None, None, 0))

# Run and time the simulation
now = time.perf_counter()

# x_t = simulate_vectorized(data, integrator_state, poses[:, 0]).
x_t, base_orientations, joint_positions, forces, link_positions = simulate(
    data, integrator_state, poses[:, 0]
)

comp_time = time.perf_counter() - now

logging.info(
    f"Running simulation with {envs_per_row**2} models took {comp_time} seconds."
)
logging.info(
    f"This corresponds to an RTF (Real Time Factor) of {(envs_per_row**2 *integration_time/1000/comp_time):.2f}"
)

In [ ]:
print(f"{model.link_names()=}")

In [ ]:
from pathlib import Path

for pose, orientation, s_t in zip(x_t, base_orientations, joint_positions):
    mj_helper.set_base_position(pose)
    mj_helper.set_base_orientation(orientation)
    # mj_helper.set_joint_positions(positions=s_t, joint_names=model.joint_names())
    recorder.record_frame()  # camera_name="pendulum_camera")

import datetime

import mediapy as media

media.show_video(recorder.frames, fps=1 / dt)

recorder.write_video(
    path=Path.cwd() / Path(f"{model.name()}_{data.velocity_representation}.mp4"),
    exist_ok=True,
)

In [ ]:
Path.cwd() / Path(f"{model.name()}_{data.velocity_representation}.mp4")

Now let's extract the data from the simulation and plot it. We expect to see the height time series of each sphere starting from a different value.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.plot(np.arange(len(x_t[:])) * dt, np.array(x_t)[:, 2])
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Height [m]")
plt.title("Trajectory of the model's base")
plt.show()

In [ ]:
forces = np.array([force for force in forces])
link_positions = np.array(link_positions)

In [ ]:
forces.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.plot(
    np.arange(len(link_positions)) * dt,
    link_positions,
    # label=["L", "R"],
)
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Link CoM")
plt.legend()
plt.show()

In [ ]:
forces[0]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.plot(
    np.arange(len(forces[:])) * dt,
    forces[:],
    label=["X", "Y", "Z", "Rx", "Ry", "Rz"],
)
plt.grid(True)
plt.xlabel("Time [s]")
plt.ylabel("Force [N]")
plt.title("Contact forces")
plt.legend()
plt.show()